<table style="width:100%;background:#fff;border-radius:8px;border-collapse:separate"><tr><td style="padding:14px 24px;text-align:left;vertical-align:middle"><a href="https://idia.uchile.cl/" target="_blank" rel="noopener" title="IDIA, Iniciativa de Datos e Inteligencia Artificial" style="justify-self:start"><img src="https://idia.uchile.cl/wp-content/uploads/2026/10/logos_logo_idia_horizontal_morado.png" alt="IDIA, Iniciativa de Datos e Inteligencia Artificial, Universidad de Chile" height="64" style="display:block"></a></td><td style="padding:14px 24px;text-align:center;vertical-align:middle"><a href="https://www.dash-ai.com/" target="_blank" rel="noopener" title="Sitio de DashAI"><img src="https://raw.githubusercontent.com/DashAISoftware/DashAI/production/docs/static/img/dashai-logo.svg" alt="DashAI" width="183" height="51" style="display:block"></a></td><td style="padding:14px 24px;text-align:right;vertical-align:middle"><a href="https://dcc.uchile.cl/" target="_blank" rel="noopener" title="Departamento de Ciencias de la Computación, Universidad de Chile" style="justify-self:end"><img src="https://raw.githubusercontent.com/DashAISoftware/DashAI/production/docs/static/img/institutions/dcc-logo.png" alt="DCC, Departamento de Ciencias de la Computación, Universidad de Chile" height="96" style="display:block"></a></td></tr></table>

# DashAI en Google Colab: entrena modelos y conversa con un modelo de lenguaje sin instalar nada en tu computador

[Pablo Badilla](https://github.com/pbadillatorrealba)<br>Tech Lead Oficina de Proyectos<br>[Iniciativa de Datos e Inteligencia Artificial](https://idia.uchile.cl/)<br>[Facultad de Ciencias Físicas y Matemáticas](https://ingenieria.uchile.cl/)<br>[Universidad de Chile](https://uchile.cl/)

**Lo que necesitas:** una cuenta de Google y un navegador (Chrome funciona mejor).

**Cuánto tiempo necesitas:** la preparación demora entre 10 y 20 minutos la primera vez, casi todo esperando las instalaciones. Cada ejemplo, unos 20 minutos más.

**Cómo se usa este material.** Son dos documentos: este cuaderno deja DashAI funcionando, y la **guía de ejercicios** (te la entregaron junto con este cuaderno) explica qué hacer dentro de DashAI. Ábrelos lado a lado, la guía en una ventana y el cuaderno en otra, para leer sin desplazarte. Los ejemplos son dos:

- **Ejemplo A:** clasificación con una tabla de datos (aprendizaje supervisado).
- **Ejemplo B:** generación de texto conversando con un modelo de lenguaje.

En este cuaderno solo haces la preparación (sección 2), descargas el archivo del Ejemplo A (sección 3) y apagas todo al terminar (sección 4).

## 1. ¿Qué es un notebook?

Lo que estás leyendo se llama *notebook* o *cuaderno*. Es un documento que mezcla texto, como este, con pequeños bloques de código llamados **celdas**.

[**Google Colab**](https://colab.research.google.com/) es un servicio gratuito de Google que te da acceso, por unas horas, a un computador en internet. Todo lo que hagamos ocurre ahí y no en el tuyo.

### Cómo ejecutar una celda

Pasa el cursor sobre la celda y haz clic en el botón **▶** que aparece a la izquierda. En las celdas que tienen un título, el botón está justo al lado del título.

Mientras la celda trabaja se ve un círculo girando; cuando termina, aparece una marca verde (✓). Espera esa marca antes de pasar a la siguiente celda. Si quieres conocer más de Colab, Google tiene una [introducción](https://colab.research.google.com/notebooks/intro.ipynb) y unas [preguntas frecuentes](https://research.google.com/colaboratory/faq.html).

<div role="note" style="background:#fff4dc;color:#1a1a1a;border-left:6px solid #c98a00;padding:10px 16px;border-radius:6px;margin:8px 0"><b>Ejecuta las celdas de arriba hacia abajo</b><br>Cada celda depende de la anterior, así que si te saltas una, la siguiente puede fallar. Si Colab te avisa que el cuaderno no fue creado por Google, elige <b>Ejecutar de todos modos</b>.</div>

## 2. Preparación

### Paso 0. Conectarse a un computador con GPU

Una GPU es un componente del computador que acelera mucho el trabajo de estos modelos. El Ejemplo A funciona sin ella, pero el Ejemplo B es bastante más rápido con una.

1. En el menú superior, haz clic en **Entorno de ejecución** y luego en **Cambiar tipo de entorno de ejecución**.
2. Elige **GPU T4** y haz clic en **Guardar**. Si la opción no está disponible (a veces Google no tiene cupo gratuito), deja la CPU (el procesador normal del computador): el Ejemplo B funcionará igual, pero con respuestas más lentas.
3. Arriba a la derecha, haz clic en **Conectar** y espera a que aparezca la marca verde.
4. Ejecuta la celda de abajo: revisa el entorno (Python, memoria, disco y GPU) y te dice qué esperar.

In [ ]:
# @title Paso 0b. Revisar el entorno { display-mode: "form" }
import os
import re
import shutil
import subprocess
import sys

ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
disco = shutil.disk_usage("/").free / 1e9
print(
    f"Python {sys.version.split()[0]} | memoria {ram:.0f} GB | disco libre {disco:.0f} GB"
)

if sys.version_info < (3, 11):  # noqa: UP036 - Colab controla su Python
    print(
        "Aviso: DashAI necesita Python 3.11 o más. Cambia el tipo de entorno o pide ayuda."
    )
if ram < 8:
    print("Aviso: hay poca memoria. El Ejemplo B puede fallar con el modelo grande.")
if disco < 10:
    print("Aviso: queda poco disco. Borra archivos del panel izquierdo de Colab.")

if shutil.which("nvidia-smi"):
    salida = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
    cuda = re.search(r"CUDA Version: ([\d.]+)", salida)
    nombre = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    ).stdout.strip()
    print(
        f"Listo: hay una GPU ({nombre}, CUDA {cuda.group(1) if cuda else '?'}). El Ejemplo B irá rápido."
    )
else:
    print(
        "No hay GPU. El Ejemplo A funciona igual; el Ejemplo B será más lento "
        "(un minuto o más por respuesta larga). Si quieres una, repite el Paso 0."
    )

### Paso 1. Instalar DashAI

Haz clic en **▶** en la celda de abajo. Demora varios minutos y va mostrando texto mientras trabaja; es normal. Cuando termine, dirá "Listo: DashAI instalado".

Si al terminar el Paso 1 o el Paso 2 aparece el botón **Reiniciar sesión**, haz clic en él, espera a que se reconecte y sigue con el paso siguiente. No hace falta repetir el que ya terminó.

In [ ]:
# @title Paso 1. Instalar DashAI { display-mode: "form" }
!pip install -q dashai==0.10.0
print("Listo: DashAI instalado.")

### Paso 2. Instalar el soporte para modelos de lenguaje

Solo se necesita para el Ejemplo B, pero conviene hacerlo ahora para esperar una sola vez. La celda debe terminar diciendo "Listo" e indica si quedó la versión con GPU o la que usa solo el procesador (más lenta). Si dice que no pudo instalar, igual puedes hacer el Ejemplo A.

In [ ]:
# @title Paso 2. Instalar soporte para modelos de lenguaje { display-mode: "form" }
import shutil
import subprocess
import sys

URL = "https://abetlen.github.io/llama-cpp-python/whl/"


def instalar(version):
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "llama-cpp-python"]
    )
    # Solo paquetes ya compilados: si no hay uno, falla enseguida en vez de compilar durante media hora.
    return (
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--only-binary=llama-cpp-python",
                "llama-cpp-python",
                "--extra-index-url",
                URL + version,
            ]
        ).returncode
        == 0
    )


def funciona():
    # La versión con GPU se instala bien aunque falte CUDA, y recién falla al importarla.
    return (
        subprocess.run(
            [sys.executable, "-c", "import llama_cpp"], capture_output=True
        ).returncode
        == 0
    )


modo = None
if shutil.which("nvidia-smi") and instalar("cu124") and funciona():
    modo = "con GPU"
elif instalar("cpu") and funciona():
    modo = "solo con procesador (el Ejemplo B será más lento)"

if modo:
    print("Listo: soporte para modelos de lenguaje instalado, " + modo + ".")
else:
    # Un paquete roto impediría encender DashAI, así que se quita.
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "llama-cpp-python"]
    )
    print("No se pudo instalar. Puedes hacer el Ejemplo A; para el B pide ayuda.")

### Paso 3. Encender DashAI

**Sobre guardar tu trabajo.** El computador que presta Colab se borra al cerrar la sesión, y con él todo lo que hayas hecho. Para conservar tus sesiones de DashAI y tus conjuntos de datos (*datasets*):

1. Marca la casilla **guardar_en_drive** de la celda antes de ejecutarla.
2. Si ya habías encendido DashAI, apágalo primero con la celda **Apagar DashAI** de la sección 4, para que el cambio tenga efecto.
3. Colab abrirá una ventana para pedir permiso a tu [Google Drive](https://drive.google.com/).

Con Drive todo puede ir algo más lento.

<div role="note" style="background:#fde9e9;color:#1a1a1a;border-left:6px solid #c0392b;padding:10px 16px;border-radius:6px;margin:8px 0"><b>Datos de pacientes</b><br>Colab funciona en servidores de Google. No subas información que identifique a pacientes (nombre, RUT, ficha, fecha de nacimiento) sin anonimizarla y sin la autorización que corresponda en tu proyecto. Para practicar, usa solo el dataset de ejemplo de este cuaderno.</div>

Haz clic en **▶** y espera. La primera vez puede tardar varios minutos. Si vuelves a ejecutar la celda mientras trabaja, no pasa nada: solo te avisará que sigue encendiéndose. Espera el mensaje "DashAI está listo".

In [ ]:
# @title Paso 3. Encender DashAI { display-mode: "form" }
guardar_en_drive = False  # @param {type:"boolean"}
import subprocess
import sys
import time
import urllib.request


def esta_activo():
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/app/", timeout=2)
        return True
    except Exception:
        return False


if esta_activo():
    print("DashAI ya estaba encendido. Puedes pasar al Paso 4.")
elif "servidor" in globals() and globals()["servidor"].poll() is None:
    print(
        "DashAI todavía se está encendiendo. Espera un par de minutos y vuelve a ejecutar esta celda."
    )
else:
    ruta = "/content/DashAI"
    if guardar_en_drive:
        from google.colab import drive

        try:
            drive.mount("/content/drive")
        except Exception:
            raise SystemExit(
                "No se pudo conectar Google Drive. Desmarca la casilla o acepta el permiso y repite."
            ) from None
        ruta = "/content/drive/MyDrive/DashAI"

    registro = open("/content/dashai.log", "w")  # noqa: SIM115 - vive entre celdas
    servidor = subprocess.Popen(
        [sys.executable, "-m", "DashAI", "--no-browser", "--local-path", ruta],
        stdout=registro,
        stderr=subprocess.STDOUT,
    )
    print("Encendiendo DashAI, espera por favor...")
    for _ in range(600):
        if servidor.poll() is not None:
            print(
                "DashAI se detuvo. Últimas líneas del registro (cópialas si necesitas pedir ayuda):"
            )
            with open("/content/dashai.log") as archivo:
                print(archivo.read()[-2000:])
            break
        if esta_activo():
            print("DashAI está listo. Pasa al Paso 4.")
            break
        time.sleep(2)
    else:
        print(
            "DashAI no respondió a tiempo. Vuelve a ejecutar esta celda y, si se repite, pide ayuda."
        )

### Paso 4. Abrir DashAI

Ejecuta esta celda solo después de ver "DashAI está listo".

- DashAI aparece en un recuadro debajo de la celda. Si lo ves pequeño, aumenta el valor de **alto** y ejecuta la celda otra vez.
- La primera vez, DashAI muestra un tutorial guiado; puedes cerrarlo con **Saltar tutorial**.
- Si el recuadro queda en blanco o lo borras, vuelve a ejecutar esta celda para mostrarlo de nuevo.
- Cuando veas DashAI en el recuadro, sigue con la guía de ejercicios.

In [ ]:
# @title Paso 4. Abrir DashAI { display-mode: "form" }
alto = 800  # @param {type:"integer"}
from google.colab import output

output.serve_kernel_port_as_iframe(8000, path="/app/", height=alto)

## 3. Archivos para los ejemplos

### Paso 5. Descargar los datos del Ejemplo A

Solo lo necesitas para el Ejemplo A. Ejecuta la celda de abajo: genera el archivo `breast_cancer.csv` (el dataset *Breast Cancer Wisconsin*, que viene con scikit-learn) y el navegador lo descarga a tu computador. Guárdalo: lo subirás a DashAI siguiendo la guía.

In [ ]:
# @title Paso 5. Descargar los datos del Ejemplo A { display-mode: "form" }
from google.colab import files
from sklearn.datasets import load_breast_cancer

datos = load_breast_cancer(as_frame=True).frame
# En scikit-learn 0 = maligno y 1 = benigno; se escriben los nombres para que se entiendan en DashAI.
datos["diagnosis"] = datos.pop("target").map({0: "malignant", 1: "benign"})
datos.to_csv("breast_cancer.csv", index=False)
print(datos.shape, datos["diagnosis"].value_counts().to_dict())
files.download("breast_cancer.csv")

## 4. Cuando termine

**Antes de apagar, descarga lo que quieras conservar.** Si no marcaste la casilla de Drive en el Paso 3, todo lo que hiciste se borra al cerrar. Si hiciste predicciones, puedes descargar tus resultados con el botón **Descargar Resultados de Predicción**.

Después ejecuta la celda de abajo para apagarlo. Por último, en el menú **Entorno de ejecución**, elige **Desconectar y eliminar el entorno de ejecución** (el texto puede variar un poco), así Google libera el computador para otras personas.

In [ ]:
# @title Apagar DashAI { display-mode: "form" }
import subprocess

# Los corchetes evitan que pkill se detenga a sí mismo al leer su propia línea de comandos
patrones = ["[D]ashAI --no-browser", "[h]uey_consumer"]
# pkill devuelve 0 solo si encontró algo que apagar
apagados = [subprocess.run(["pkill", "-f", x]).returncode == 0 for x in patrones]
print("DashAI apagado." if any(apagados) else "DashAI no estaba encendido.")

## 5. Si algo no funciona

### Preparación

**Una celda se queda con el círculo girando mucho rato.** Los pasos 1 y 3 pueden tardar varios minutos. Si pasan más de 20 minutos, detén la celda con el botón cuadrado y vuelve a ejecutarla.

**El Paso 3 dice "DashAI se detuvo", o DashAI se apagó (Colab se reinició o la sesión caducó).** Si el mensaje dice "No module named DashAI", falta el Paso 1 o Colab se reinició: repite los pasos 1 al 3 en orden. Si el registro menciona `llama_cpp` o `libcudart`, repite el Paso 2 y luego el 3. Si no, repite el Paso 3 y luego el 4. Si marcaste **guardar_en_drive**, tus sesiones y datasets siguen ahí. Si no, se perdieron: vuelve a subir `breast_cancer.csv` (queda en tu computador) y repite el ejemplo de la guía.

**El recuadro del Paso 4 queda en blanco o muestra un error de Google (403 o 404).** Revisa que en el navegador esté la misma cuenta de Google con la que abriste este cuaderno y que no bloquees las cookies de terceros. Tampoco funciona si ejecutaste el Paso 4 antes de que DashAI estuviera listo. No intentes abrir la dirección del recuadro en una pestaña aparte: Google la rechaza.

**El recuadro de DashAI quedó en blanco o lo borraste.** DashAI sigue funcionando en Colab. Ejecuta de nuevo el Paso 4 y tu trabajo estará ahí. Si sigue sin aparecer, ejecuta el Paso 3: si dice "DashAI ya estaba encendido", basta con volver al Paso 4.

### Datos del Ejemplo A

**El archivo `breast_cancer.csv` no se descarga.** El navegador puede haber bloqueado la descarga: revisa si aparece un aviso en la barra de direcciones y permítela. Si no, ejecuta de nuevo la celda y busca el archivo en la carpeta de descargas.

### En cualquier momento

**Aparece un mensaje en rojo en una celda.** Selecciona con el mouse las últimas líneas, cópialas (Ctrl+C) y envíalas a quien te entregó este cuaderno.

**Se acabó el tiempo de la sesión.** Colab gratuito corta tras unas horas de uso o de inactividad. Vuelve a ejecutar el cuaderno desde el Paso 0.


Si el problema ocurre dentro de DashAI (un botón que no responde, un modelo que no se descarga), mira la sección *Si algo no funciona* de la guía de ejercicios.